# State Management with StateMachine
This notebook demonstrates how to work with state management using a StateMachine implementation. We'll explore how to create, manage, and control workflow states in a structured way.

## What we'll learn:
- Basic state machine concepts and implementation
- Creating and connecting workflow steps
- Managing state transitions and data flow
- Working with routing and loops in state machines
- Understanding state snapshots and execution flow

### Setup

In [1]:
from typing import TypedDict
from lib.state_machine import (
    StateMachine,
    Step,
    EntryPoint,
    Termination,
)

## Basic State Machine Concepts
Let's start with a simple example that demonstrates the core concepts of our state machine:
1. Defining state schema
2. Creating steps
3. Connecting steps
4. Running the workflow

**Creating the Schema and the State Machine**

In [2]:
class Schema(TypedDict):
    """Schema defining the structure of our state.
    
    Attributes:
        input: The input value to process
        output: The processed output value
    """
    input: int
    output: int

In [3]:
# Create our state machine instance
workflow = StateMachine(Schema)

**Defining the logic for Steps**

In [5]:
def step_input(state: Schema) -> Schema:
    """First step: Increment the input value.
    
    Args:
        state: Current state containing input value
        
    Returns:
        Updated state with incremented value in output
    """
    return {"output": state["input"] + 1, "random": 10}



In [6]:
def step_double(state: Schema) -> Schema:
    """Second step: Double the previous output.
    
    Args:
        state: Current state containing output from previous step
        
    Returns:
        Updated state with doubled output value
    """
    return {"output": state["output"] * 2}


**Creating and Connecting Steps**

In [7]:
entry = EntryPoint()
s1 = Step("input", step_input)
s2 = Step("double", step_double)
termination = Termination()

In [8]:
workflow.add_steps([entry, s1, s2, termination])

In [9]:
workflow.connect(entry, s1)
workflow.connect(s1, s2)
workflow.connect(s2, termination)

In [10]:
workflow.transitions

{'__entry__': [Transition('__entry__' -> ['input'])],
 'input': [Transition('input' -> ['double'])],
 'double': [Transition('double' -> ['__termination__'])]}

**Running the Workflow**

In [11]:
initial_state = {"input": 4}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: __entry__
[StateMachine] Executing step: input
[StateMachine] Executing step: double
[StateMachine] Terminating: __termination__


Run('65984a2f-dfbc-4664-9afa-0f16bed7bc45')

In [12]:
run_object.snapshots

[Snapshot('bbe26a7b-bae4-4e41-a180-c639ab3628de') @ [2026-01-09 18:50:15.921451]: __entry__.State({'input': 4}),
 Snapshot('d107eade-ef99-4d54-9381-c7084e81d069') @ [2026-01-09 18:50:15.921521]: input.State({'input': 4, 'output': 5}),
 Snapshot('cda33b11-ea9e-4d37-b369-b1f17a4eae47') @ [2026-01-09 18:50:15.921582]: double.State({'input': 4, 'output': 10})]

## Advanced State Management: Routing and Loops
Now we'll explore more complex state management patterns including:
- Conditional routing between steps
- Creating loops in the workflow
- Managing state through multiple iterations

In [13]:
class CounterSchema(TypedDict):
    """Schema for a counter-based workflow.
    
    Attributes:
        count: Current counter value
        max_value: Maximum value before termination
    """
    count: int
    max_value: int

In [14]:
workflow = StateMachine(CounterSchema)

In [15]:
def increment_counter(state: CounterSchema) -> CounterSchema:
    """Increment the counter value.
    
    Args:
        state: Current state with counter value
        
    Returns:
        Updated state with incremented counter
    """
    return {"count": state["count"] + 1}

In [16]:
# Create steps
entry = EntryPoint()
increment = Step("increment", increment_counter)
termination = Termination()

In [17]:
workflow.add_steps([entry, increment, termination])

In [18]:
# Router logic
def check_counter(state: CounterSchema) -> Step:
    """Determine next step based on counter value.
    
    Args:
        state: Current state with counter and max value
        
    Returns:
        Next step to execute (increment or terminate)
    """
    if state["count"] >= state["max_value"]:
        return termination
    return increment

In [19]:
# Connect steps with a loop in increment
workflow.connect(entry, increment)
workflow.connect(increment, [increment, termination], check_counter)

In [20]:
workflow.transitions

{'__entry__': [Transition('__entry__' -> ['increment'])],
 'increment': [Transition('increment' -> ['increment', '__termination__'])]}

In [21]:
initial_state = {"count": 0, "max_value": 3}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: __entry__
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Terminating: __termination__


Run('8ac71d92-56c9-4d1e-8167-5c743e4eeddd')

In [22]:
run_object.snapshots

[Snapshot('f8a91515-f571-4a01-a8e7-051e70a1caf7') @ [2026-01-09 18:53:33.667192]: __entry__.State({'count': 0, 'max_value': 3}),
 Snapshot('a7cb5bec-087b-4011-92da-912298c00389') @ [2026-01-09 18:53:33.667259]: increment.State({'count': 1, 'max_value': 3}),
 Snapshot('0bb659f4-cf88-4349-839c-6ef5c45d3013') @ [2026-01-09 18:53:33.667319]: increment.State({'count': 2, 'max_value': 3}),
 Snapshot('83853b32-1728-4fc0-8b66-044e3d24aadc') @ [2026-01-09 18:53:33.667377]: increment.State({'count': 3, 'max_value': 3})]

In [23]:
run_object.get_final_state()

{'count': 3, 'max_value': 3}